# MLP Experience Replay with Confidently Correct Memory
Train an MLPClassifier with year-wise incremental scaling and experience replay prioritizing confidently correct samples.

In [1]:
import copy
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore')

## 1. Load Data and Splits

In [2]:
ds_path = Path('.') / 'training_data_with_features_plus_monthly_indices.zarr'
print(f'Loading data from {ds_path}...')
ds = xr.open_dataset(ds_path, engine='zarr')
print('Data loaded')

split_path = Path('.') / 'data_split.npz'
print(f'Loading split from {split_path}...')
split_data = np.load(split_path)
train_pixel_indices = split_data['train_pixel_indices']
val_pixel_indices = split_data['val_pixel_indices']
test_pixel_indices = split_data['test_pixel_indices']
print('Split loaded')

print('Dataset info:')
print(f'  Total pixels: {len(ds.pixel)}')
print(f'  Total years: {len(ds.year)}')
print(f'  Train pixels: {len(train_pixel_indices)}')
print(f'  Val pixels: {len(val_pixel_indices)}')
print(f'  Test pixels: {len(test_pixel_indices)}')

Loading data from training_data_with_features_plus_monthly_indices.zarr...


Data loaded
Loading split from data_split.npz...
Split loaded
Dataset info:
  Total pixels: 8155205
  Total years: 7
  Train pixels: 5597776
  Val pixels: 1273437
  Test pixels: 1283992


## 2. Feature Engineering

In [3]:
def prepare_raw_features_for_year(ds, pixel_indices, year_idx, s2_mean_per_pixel=None, dtype=np.float32):
    """Extract and clean raw (unscaled) features for one year. Year 0 is skipped by design."""
    if year_idx == 0:
        return np.empty((0, 0), dtype=dtype), np.empty((0,), dtype=np.int64)

    if s2_mean_per_pixel is None:
        s2_all_years = ds['s2_bands'].isel(pixel=pixel_indices).values
        s2_mean_per_pixel = np.nanmean(s2_all_years, axis=1)

    ds_subset = ds.isel(pixel=pixel_indices, year=year_idx)
    s2_features = ds_subset['s2_bands'].values
    if np.isnan(s2_features).any():
        s2_features = np.where(np.isnan(s2_features), s2_mean_per_pixel, s2_features)

    dem_features = ds_subset['dem'].values.reshape(-1, 1)
    ndvi_features = ds_subset['ndvi'].values.reshape(-1, 1)
    ndwi_features = ds_subset['ndwi'].values.reshape(-1, 1)

    ds_prev = ds.isel(pixel=pixel_indices, year=year_idx - 1)
    ndvi_last_year = np.where(np.isnan(ds_prev['ndvi'].values.reshape(-1, 1)), 0, ds_prev['ndvi'].values.reshape(-1, 1))
    ndwi_last_year = np.where(np.isnan(ds_prev['ndwi'].values.reshape(-1, 1)), 0, ds_prev['ndwi'].values.reshape(-1, 1))

    required_last_year_bands = ['B04', 'B03', 'B06']
    band_to_idx = {band: i for i, band in enumerate(ds['s2_band'].values)}
    missing_bands = [band for band in required_last_year_bands if band not in band_to_idx]
    if missing_bands:
        raise ValueError(f"Missing required S2 bands for last-year features: {missing_bands}")

    last_year_s2_features = []
    for band in required_last_year_bands:
        band_values = ds_prev['s2_bands'].sel(s2_band=band).values
        band_values = np.where(np.isnan(band_values), 0, band_values)
        last_year_s2_features.append(band_values.reshape(-1, 1))

    # Keep feature order aligned with MLP training notebook.
    features_list = [
        s2_features,
        dem_features,
        ndvi_features,
        ndwi_features,
        ndvi_last_year,
        ndwi_last_year,
        *last_year_s2_features,
    ]

    if 'nbr' in ds.data_vars:
        features_list.append(ds_subset['nbr'].values.reshape(-1, 1))

    if year_idx > 0 and 'ndvi_delta' in ds.data_vars:
        delta_year_idx = year_idx - 1
        ds_delta = ds.isel(pixel=pixel_indices, year=delta_year_idx)
        features_list.append(ds_delta['ndvi_delta'].values.reshape(-1, 1))
        features_list.append(ds_delta['ndwi_delta'].values.reshape(-1, 1))
        if 'nbr_delta' in ds.data_vars:
            features_list.append(ds_delta['nbr_delta'].values.reshape(-1, 1))

    for var_name in ['years_since_last_disturbance', 'log_years_since_last_disturbance', 'ever_disturbed']:
        if var_name in ds.data_vars:
            features_list.append(ds_subset[var_name].values.reshape(-1, 1))

    yearly_index_feature_names = [
        'ndvi_cv_year',
        'ndvi_max_m2m_drop_year',
        'ndvi_max_year',
        'ndvi_min_year',
        'ndvi_std_year',
        'ndwi_cv_year',
        'ndwi_max_m2m_drop_year',
        'ndwi_max_year',
        'ndwi_min_year',
        'ndwi_std_year',
    ]
    for feature_name in yearly_index_feature_names:
        if feature_name in ds.data_vars:
            features_list.append(ds_subset[feature_name].values.reshape(-1, 1))

    X = np.concatenate(features_list, axis=1)
    y = ds_subset['disturbances'].values

    valid_label_mask = np.isin(y, [0, 1])
    X = X[valid_label_mask]
    y = y[valid_label_mask]

    nan_mask = ~np.isnan(X).any(axis=1)
    X_clean = X[nan_mask]
    y_clean = y[nan_mask]

    if len(X_clean) == 0:
        feature_dim = X.shape[1] if X.ndim == 2 and X.shape[0] > 0 else 0
        return np.empty((0, feature_dim), dtype=dtype), np.empty((0,), dtype=np.int64)

    return X_clean.astype(dtype, copy=False), y_clean.astype(np.int64, copy=False)


def prepare_features_for_year(ds, pixel_indices, year_idx, scaler=None, scaler_mode='auto', s2_mean_per_pixel=None, dtype=np.float32):
    """Extract, clean, and optionally scale features for one year. Year 0 is skipped by design."""
    valid_scaler_modes = {'auto', 'fit', 'partial_fit', 'transform', 'none'}
    if scaler_mode not in valid_scaler_modes:
        raise ValueError(f"Invalid scaler_mode '{scaler_mode}'. Valid options: {sorted(valid_scaler_modes)}")

    X_clean, y_clean = prepare_raw_features_for_year(
        ds,
        pixel_indices,
        year_idx,
        s2_mean_per_pixel=s2_mean_per_pixel,
        dtype=dtype,
    )

    if len(X_clean) == 0:
        return X_clean, y_clean, scaler

    if scaler_mode == 'none':
        return X_clean, y_clean, scaler

    if scaler is None:
        scaler = StandardScaler()

    if scaler_mode == 'auto':
        if hasattr(scaler, 'mean_'):
            X_clean = scaler.transform(X_clean)
        else:
            X_clean = scaler.fit_transform(X_clean)
    elif scaler_mode == 'fit':
        X_clean = scaler.fit_transform(X_clean)
    elif scaler_mode == 'partial_fit':
        scaler.partial_fit(X_clean)
        X_clean = scaler.transform(X_clean)
    elif scaler_mode == 'transform':
        if not hasattr(scaler, 'mean_'):
            raise ValueError("Scaler must be fitted before using scaler_mode='transform'.")
        X_clean = scaler.transform(X_clean)

    return X_clean, y_clean, scaler

## 3. Initialize Class Weights and MLP

In [4]:
print('Precomputing raw yearly features for train/validation splits...')
n_years = len(ds.year)

def precompute_yearly_raw_cache(ds, pixel_indices, n_years, split_name):
    s2_all_years = ds['s2_bands'].isel(pixel=pixel_indices).values
    s2_mean_per_pixel = np.nanmean(s2_all_years, axis=1)

    cache = {}
    empty_years = 0
    for year_idx in tqdm(range(1, n_years), desc=f'Precompute {split_name}'):
        X_raw, y_raw = prepare_raw_features_for_year(
            ds,
            pixel_indices,
            year_idx,
            s2_mean_per_pixel=s2_mean_per_pixel,
            dtype=np.float32,
        )
        cache[year_idx] = (X_raw, y_raw)
        if len(y_raw) == 0:
            empty_years += 1

    print(
        f"{split_name}: cached {len(cache)} years, empty years={empty_years}, "
        f"sample feature dim={next((x.shape[1] for x, y in cache.values() if len(y) > 0), 0)}"
    )
    return cache

train_feature_cache = precompute_yearly_raw_cache(ds, train_pixel_indices, n_years, 'train')
val_feature_cache = precompute_yearly_raw_cache(ds, val_pixel_indices, n_years, 'validation')

classes = np.array([0, 1])

model = MLPClassifier(
    hidden_layer_sizes=(64,),
    activation='relu',
    alpha=0.0001,
    random_state=42,
    solver='adam',
    learning_rate='adaptive',
    max_iter=1,
    learning_rate_init=0.001,
    warm_start=False,
    verbose=False,
)
print('MLP initialized')

Precomputing raw yearly features for train/validation splits...


Precompute train:   0%|          | 0/6 [00:00<?, ?it/s]

train: cached 6 years, empty years=0, sample feature dim=32


Precompute validation:   0%|          | 0/6 [00:00<?, ?it/s]

validation: cached 6 years, empty years=0, sample feature dim=32
MLP initialized


## 4. Replay-Enabled Online Training

### Replay Sampling Strategy (Confidently Correct + Random)
For each training year, replay samples are split into two parts:
- **20% confidently correct memory replay**
- **80% uniform random replay**

Confidently correct memory is sampled as follows:
1. Identify all previous years with available replay samples.
2. Split the confidently correct replay target as evenly as possible across these previous years.
3. For each previous year, run current-model probability predictions on all that year's replay candidates.
4. Compute the **optimal F1 threshold** for that year from probabilities and labels (threshold grid search).
5. Determine correct classifications where predicted class matches the true class: $y_{pred} == y_{true}$, where $y_{pred} = 1$ if $p_i \ge t^*$ else $0$, and $t^*$ is the optimal F1 threshold.
6. For correctly classified samples, compute weights based on the probability of the true class (confidence):

$$w_i = 1.0 - |y_i - p_i|$$

For incorrectly classified samples, set weight to 0.0.

7. Sample from that year's candidates using these weights.

The random part is sampled uniformly from remaining replay candidates to avoid overlap with confidently correct-selected samples.

In [5]:
import json
from datetime import datetime

REPLAY_RATIOS = [0.2, 0.3, 0.4, 0.5]
REPLAY_ENABLED = True
REPLAY_RANDOM_STATE = 42
CONFIDENTLY_CORRECT_REPLAY_FRACTION = 0.2
THRESHOLD_GRID = np.linspace(0.0, 1.0, 201)
EPSILON = 1e-12

WEIGHT_POLICY_NAME = 'post_sampling_replay_class_weights_v1'
REPLAY_WEIGHT_FALLBACK = 'smoothed_single_class'
REPLAY_POSITIVE_LABEL = 1

CHUNK_SIZE = 50000
MAX_EPOCHS = 15
PATIENCE = 3
MIN_DELTA = 0.0005

if REPLAY_WEIGHT_FALLBACK not in {'smoothed_single_class'}:
    raise ValueError(f'Unsupported REPLAY_WEIGHT_FALLBACK: {REPLAY_WEIGHT_FALLBACK}')

CHECKPOINT_DIR = Path('.') / 'training_checkpoints_mlp_experience_replay_confidently_correct_memory'
ALL_HISTORIES_FILE = CHECKPOINT_DIR / 'mlp_replay_all_training_histories_confidently_correct_memory.pkl'
COMPLETION_STATUS_FILE = CHECKPOINT_DIR / 'mlp_replay_completion_status_confidently_correct_memory.json'
TRAINING_LOG_FILE = CHECKPOINT_DIR / 'mlp_replay_training_log_confidently_correct_memory.txt'
COMBINED_HISTORY_FILENAME = 'mlp_classifier_history_prevyears_monthly_features_incremental_scaler_experience_replay_confidently_correct_memory_all_ratios.csv'

CHECKPOINT_DIR.mkdir(exist_ok=True)

if 'train_feature_cache' not in globals() or 'val_feature_cache' not in globals() or not train_feature_cache or not val_feature_cache:
    raise ValueError('Raw feature caches not found or empty. Run Cell 8 first to precompute yearly features.')

def create_empty_training_history():
    return {
        'year': [],
        'train_accuracy': [],
        'train_precision': [],
        'train_recall': [],
        'train_f1': [],
        'val_accuracy': [],
        'val_precision': [],
        'val_recall': [],
        'val_f1': [],
        'val_roc_auc': [],
        'val_pr_auc': [],
        'replay_pool_size': [],
        'replay_target_size': [],
        'replay_used_size': [],
        'confidently_correct_target_size': [],
        'confidently_correct_used_size': [],
        'random_target_size': [],
        'random_used_size': [],
        'threshold_mean': [],
        'weight_policy': [],
        'year_positive_rate': [],
        'replay_actual_positive_rate': [],
        'replay_pool_positive_rate': [],
        'current_class_weight_0': [],
        'current_class_weight_1': [],
        'replay_class_weight_0': [],
        'replay_class_weight_1': [],
        'replay_weight_fallback': [],
    }

def load_all_training_histories(path):
    if path.exists():
        with open(path, 'rb') as f:
            data = pickle.load(f)
        if isinstance(data, dict):
            return data
    return {}

def save_all_training_histories(path, all_histories):
    with open(path, 'wb') as f:
        pickle.dump(all_histories, f)

def load_completion_status(path):
    default_status = {'completed_ratios': [], 'completed_years': {}, 'weight_policy_by_ratio': {}}
    if path.exists():
        with open(path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        if isinstance(data, dict):
            if 'completed_ratios' not in data:
                data['completed_ratios'] = []
            if 'completed_years' not in data:
                data['completed_years'] = {}
            if 'weight_policy_by_ratio' not in data:
                data['weight_policy_by_ratio'] = {}
            return data
    return default_status

def save_completion_status(path, status):
    status['completed_ratios'] = sorted(list(set(status.get('completed_ratios', []))))
    cleaned_completed_years = {}
    for key, years in status.get('completed_years', {}).items():
        cleaned_completed_years[key] = sorted(list(set(int(y) for y in years)))
    status['completed_years'] = cleaned_completed_years

    with open(path, 'w', encoding='utf-8') as f:
        json.dump(status, f, indent=2)

def append_training_log(path, message):
    timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    with open(path, 'a', encoding='utf-8') as f:
        f.write(f'[{timestamp}] {message}\n')

def format_ratio_key(replay_ratio):
    return f'RR_{replay_ratio:.1f}'

def build_mlp_model():
    return MLPClassifier(
        hidden_layer_sizes=(64,),
        activation='relu',
        alpha=0.0001,
        random_state=42,
        solver='adam',
        learning_rate='adaptive',
        max_iter=1,
        learning_rate_init=0.001,
        warm_start=False,
        verbose=False,
    )

def get_cached_raw_year(cache, year_idx):
    if year_idx not in cache:
        return np.empty((0, 0), dtype=np.float32), np.empty((0,), dtype=np.int64)
    return cache[year_idx]

def compute_year_positive_rate(y_batch, positive_label=1):
    if len(y_batch) == 0:
        return np.nan
    return float(np.mean(y_batch == positive_label))

def compute_binary_class_weights(y_batch, classes=np.array([0, 1]), fallback_mode='smoothed_single_class'):
    y_batch = np.asarray(y_batch, dtype=np.int64)
    if len(y_batch) == 0:
        return {0: 1.0, 1: 1.0}, 'empty_uniform'

    unique_labels = set(np.unique(y_batch).tolist())
    if not unique_labels.issubset({0, 1}):
        raise ValueError(f'Expected binary labels in {{0, 1}}, got {sorted(unique_labels)}')

    if len(unique_labels) == 2:
        class_weights_array = compute_class_weight('balanced', classes=classes, y=y_batch)
        class_weight_dict = {classes[i]: float(class_weights_array[i]) for i in range(len(classes))}
        return class_weight_dict, 'balanced'

    if fallback_mode != 'smoothed_single_class':
        raise ValueError(f'Unsupported fallback_mode: {fallback_mode}')

    n_samples = float(len(y_batch))
    positive_count = float(np.sum(y_batch == 1))
    positive_rate_smoothed = (positive_count + 1.0) / (n_samples + 2.0)
    class_weight_1 = 1.0 / (2.0 * positive_rate_smoothed)
    class_weight_0 = 1.0 / (2.0 * (1.0 - positive_rate_smoothed))
    class_weight_dict = {0: float(class_weight_0), 1: float(class_weight_1)}

    present_label = int(next(iter(unique_labels)))
    return class_weight_dict, f'smoothed_single_class_present_{present_label}'

def compute_optimal_f1_threshold(y_true, y_proba, threshold_grid, default_threshold=0.5):
    if len(y_true) == 0 or len(np.unique(y_true)) < 2:
        return float(default_threshold)

    from sklearn.metrics import precision_recall_curve
    precisions, recalls, thresholds = precision_recall_curve(y_true, y_proba)
    f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-12)
    best_idx = np.argmax(f1_scores)
    if best_idx < len(thresholds):
        best_threshold = float(thresholds[best_idx])
    else:
        best_threshold = float(default_threshold)
    return best_threshold

def weighted_choice_without_replacement(indices, weights, sample_size, rng):
    if sample_size <= 0 or len(indices) == 0:
        return np.empty((0,), dtype=np.int64)

    sample_size = min(int(sample_size), len(indices))
    weights = np.asarray(weights, dtype=np.float64)
    weights = np.where(np.isnan(weights) | (weights < 0), 0.0, weights)
    total_weight = float(weights.sum())

    if total_weight <= 0:
        selected_positions = rng.choice(len(indices), size=sample_size, replace=False)
    else:
        non_zero_mask = weights > 0
        n_non_zero = np.sum(non_zero_mask)
        if n_non_zero < sample_size:
            chosen_non_zero = np.where(non_zero_mask)[0]
            remaining_needed = sample_size - n_non_zero
            zero_indices = np.where(~non_zero_mask)[0]
            chosen_zero = rng.choice(zero_indices, size=remaining_needed, replace=False)
            selected_positions = np.concatenate([chosen_non_zero, chosen_zero])
            rng.shuffle(selected_positions)
        else:
            prob = weights / total_weight
            selected_positions = rng.choice(len(indices), size=sample_size, replace=False, p=prob)

    return np.asarray(indices, dtype=np.int64)[selected_positions]

def sample_confidently_correct_replay_indices(
    model,
    X_replay_pool,
    y_replay_pool,
    replay_year_spans,
    confident_target_size,
    rng,
    threshold_grid,
):
    if confident_target_size <= 0 or len(replay_year_spans) == 0:
        return np.empty((0,), dtype=np.int64), []

    n_years_with_data = len(replay_year_spans)
    base_quota = confident_target_size // n_years_with_data
    remainder = confident_target_size % n_years_with_data

    per_year_targets = []
    for idx, year_span in enumerate(replay_year_spans):
        year_idx, start, end = year_span
        year_target = base_quota + (1 if idx < remainder else 0)
        year_size = end - start
        per_year_targets.append((year_idx, start, end, min(year_target, year_size)))

    selected_indices = []
    threshold_values = []

    for year_idx, start, end, year_target in per_year_targets:
        if year_target <= 0:
            continue

        year_indices = np.arange(start, end, dtype=np.int64)
        y_year = y_replay_pool[start:end]
        y_proba_year = model.predict_proba(X_replay_pool[start:end])[:, 1]
        threshold = compute_optimal_f1_threshold(
            y_true=y_year,
            y_proba=y_proba_year,
            threshold_grid=threshold_grid,
            default_threshold=0.5,
        )
        threshold_values.append(threshold)

        y_pred_year = (y_proba_year >= threshold).astype(int)
        correct_mask = (y_pred_year == y_year)

        # weights are confidence of correct prediction (probability of true class) for correct samples, 0 for incorrect
        confident_weights = np.where(correct_mask, 1.0 - np.abs(y_year - y_proba_year), 0.0)

        chosen = weighted_choice_without_replacement(
            indices=year_indices,
            weights=confident_weights,
            sample_size=year_target,
            rng=rng,
        )
        if len(chosen) > 0:
            selected_indices.append(chosen)

    if selected_indices:
        confident_indices = np.concatenate(selected_indices).astype(np.int64, copy=False)
    else:
        confident_indices = np.empty((0,), dtype=np.int64)

    shortfall = int(confident_target_size) - len(confident_indices)
    if shortfall > 0:
        all_indices = np.arange(len(y_replay_pool), dtype=np.int64)
        remaining_indices = np.setdiff1d(all_indices, confident_indices, assume_unique=False)
        if len(remaining_indices) > 0:
            top_up = rng.choice(remaining_indices, size=min(shortfall, len(remaining_indices)), replace=False)
            confident_indices = np.concatenate([confident_indices, top_up.astype(np.int64, copy=False)])

    return confident_indices, threshold_values

def sample_random_replay_indices(replay_pool_size, random_target_size, excluded_indices, rng):
    if random_target_size <= 0 or replay_pool_size <= 0:
        return np.empty((0,), dtype=np.int64)

    all_indices = np.arange(replay_pool_size, dtype=np.int64)
    available_indices = np.setdiff1d(all_indices, excluded_indices, assume_unique=False)
    if len(available_indices) == 0:
        return np.empty((0,), dtype=np.int64)

    take = min(int(random_target_size), len(available_indices))
    return rng.choice(available_indices, size=take, replace=False).astype(np.int64, copy=False)

all_training_histories = load_all_training_histories(ALL_HISTORIES_FILE)
completion_status = load_completion_status(COMPLETION_STATUS_FILE)
completion_status.setdefault('weight_policy_by_ratio', {})

n_years = len(ds.year)
year_values = ds.year.values
year_value_to_idx = {int(y): idx for idx, y in enumerate(year_values)}
all_target_year_values = [int(year_values[idx]) for idx in range(1, n_years)]

print(f'Checkpoint directory: {CHECKPOINT_DIR.resolve()}')
print(f'Replay ratios: {REPLAY_RATIOS}')
print(f'Weight policy: {WEIGHT_POLICY_NAME}, replay_weight_fallback={REPLAY_WEIGHT_FALLBACK}')
append_training_log(TRAINING_LOG_FILE, f'Started training run for ratios: {REPLAY_RATIOS}')

for ratio_idx, replay_ratio in enumerate(REPLAY_RATIOS):
    ratio_key = format_ratio_key(replay_ratio)
    output_suffix = f'incremental_scaler_experience_replay_confidently_correct_memory_{ratio_key}'
    models_dir_name = f'models_mlp_prevyears_monthly_features_{output_suffix}'
    scaler_file_template = f'scaler_year_{{year}}_mlp_prevyears_monthly_features_{output_suffix}.pkl'
    model_file_template = f'model_year_{{year}}_mlp_prevyears_monthly_features_{output_suffix}.pkl'
    final_scaler_filename = f'scaler_final_mlp_prevyears_monthly_features_{output_suffix}.pkl'
    final_model_filename = f'mlp_classifier_model_prevyears_monthly_features_{output_suffix}.pkl'
    history_filename = f'mlp_classifier_history_prevyears_monthly_features_{output_suffix}.csv'

    models_dir = Path('.') / models_dir_name
    models_dir.mkdir(exist_ok=True)

    expected_policy_config = {
        'weight_policy': WEIGHT_POLICY_NAME,
        'replay_weight_fallback': REPLAY_WEIGHT_FALLBACK,
        'weight_source': 'current_year_and_post_sampling_replay',
    }
    completion_status['weight_policy_by_ratio'][ratio_key] = expected_policy_config

    if ratio_key in completion_status.get('completed_ratios', []):
        print(f'[{ratio_key}] already completed. Skipping ratio.')
        append_training_log(TRAINING_LOG_FILE, f'[{ratio_key}] skipped (already completed).')
        continue

    training_history = all_training_histories.get(ratio_key, create_empty_training_history())
    for key in create_empty_training_history().keys():
        training_history.setdefault(key, [])

    completed_years = set(int(y) for y in completion_status.get('completed_years', {}).get(ratio_key, []))
    completed_years.update(int(y) for y in training_history.get('year', []))
    completion_status.setdefault('completed_years', {})[ratio_key] = sorted(list(completed_years))

    replay_rng = np.random.default_rng(REPLAY_RANDOM_STATE + ratio_idx)

    model = None
    incremental_scaler = None
    start_year_idx = 1

    if completed_years:
        resume_candidate_years = sorted(completed_years, reverse=True)
        resumed = False
        for resume_year in resume_candidate_years:
            year_model_path = models_dir / model_file_template.format(year=resume_year)
            year_scaler_path = models_dir / scaler_file_template.format(year=resume_year)
            if year_model_path.exists() and year_scaler_path.exists() and resume_year in year_value_to_idx:
                with open(year_model_path, 'rb') as f:
                    model = pickle.load(f)
                with open(year_scaler_path, 'rb') as f:
                    incremental_scaler = pickle.load(f)
                start_year_idx = year_value_to_idx[resume_year] + 1
                resumed = True
                print(f'[{ratio_key}] resuming from year {resume_year}; continuing at index {start_year_idx}.')
                append_training_log(TRAINING_LOG_FILE, f'[{ratio_key}] resumed from year {resume_year}.')
                break

        if resumed:
            # Clean up completed_years and training_history for years after the resumed year
            completed_years = {y for y in completed_years if y <= resume_year}
            if training_history.get('year'):
                keep_indices = [i for i, y in enumerate(training_history['year']) if y <= resume_year]
                for key in training_history.keys():
                    if isinstance(training_history[key], list):
                        training_history[key] = [training_history[key][i] for i in keep_indices]
        else:
            print(f'[{ratio_key}] checkpoint artifacts missing/inconsistent. Restarting this ratio from scratch.')
            append_training_log(TRAINING_LOG_FILE, f'[{ratio_key}] restart due to missing/inconsistent checkpoint artifacts.')
            completed_years = set()
            completion_status['completed_years'][ratio_key] = []
            training_history = create_empty_training_history()

    if model is None:
        model = build_mlp_model()
    if incremental_scaler is None:
        incremental_scaler = StandardScaler()

    print(f'[{ratio_key}] Model output directory: {models_dir.resolve()}')
    print(f'[{ratio_key}] Years to train: 1..{n_years - 1} (year 0 skipped)')

    for year_idx in tqdm(range(start_year_idx, n_years), desc=f'{ratio_key} by year'):
        year_val = int(year_values[year_idx])

        if year_val in completed_years:
            continue

        X_train_raw, y_train_batch = get_cached_raw_year(train_feature_cache, year_idx)
        X_val_raw, y_val_batch = get_cached_raw_year(val_feature_cache, year_idx)

        if len(X_train_raw) == 0 or len(X_val_raw) == 0:
            print(f'[{ratio_key}] Year {year_val}: skipped (empty after filtering)')
            completed_years.add(year_val)
            completion_status['completed_years'][ratio_key] = sorted(list(completed_years))
            save_completion_status(COMPLETION_STATUS_FILE, completion_status)
            append_training_log(TRAINING_LOG_FILE, f'[{ratio_key}] year {year_val} skipped (empty after filtering).')
            continue

        year_positive_rate = compute_year_positive_rate(y_train_batch, positive_label=REPLAY_POSITIVE_LABEL)
        current_class_weight_dict, current_weight_fallback = compute_binary_class_weights(
            y_train_batch,
            classes=classes,
            fallback_mode=REPLAY_WEIGHT_FALLBACK,
        )

        incremental_scaler.partial_fit(X_train_raw)
        X_train_batch = incremental_scaler.transform(X_train_raw)
        X_val_batch = incremental_scaler.transform(X_val_raw)

        scaler_checkpoint = copy.deepcopy(incremental_scaler)
        year_scaler_path = models_dir / scaler_file_template.format(year=year_val)
        with open(year_scaler_path, 'wb') as f:
            pickle.dump(scaler_checkpoint, f)

        n_samples = len(X_train_batch)
        replay_target_size = int(n_samples * replay_ratio) if REPLAY_ENABLED else 0

        replay_X_parts = []
        replay_y_parts = []
        replay_year_spans = []
        replay_cursor = 0

        if REPLAY_ENABLED and year_idx > 1:
            for past_year_idx in range(1, year_idx):
                X_past_raw, y_past = get_cached_raw_year(train_feature_cache, past_year_idx)
                if len(X_past_raw) > 0:
                    X_past = incremental_scaler.transform(X_past_raw)
                    replay_X_parts.append(X_past)
                    replay_y_parts.append(y_past)
                    span_len = len(y_past)
                    replay_year_spans.append((past_year_idx, replay_cursor, replay_cursor + span_len))
                    replay_cursor += span_len

        if replay_X_parts:
            X_replay_pool = np.vstack(replay_X_parts)
            y_replay_pool = np.concatenate(replay_y_parts)
        else:
            X_replay_pool = np.empty((0, X_train_batch.shape[1]), dtype=X_train_batch.dtype)
            y_replay_pool = np.empty((0,), dtype=y_train_batch.dtype)

        replay_pool_size = len(y_replay_pool)
        replay_used_size = min(replay_target_size, replay_pool_size) if REPLAY_ENABLED else 0
        replay_actual_positive_rate = np.nan
        replay_pool_positive_rate = (
            float(np.mean(y_replay_pool == REPLAY_POSITIVE_LABEL))
            if replay_pool_size > 0
            else np.nan
        )

        if hasattr(model, 'n_features_in_') and int(model.n_features_in_) != int(X_train_batch.shape[1]):
            raise ValueError(
                f'Feature count mismatch at year {year_val}: model expects {model.n_features_in_}, got {X_train_batch.shape[1]}'
            )

        best_val_pr_auc = -np.inf
        patience_counter = 0
        best_model_state = None
        confident_used_size = 0
        random_used_size = 0
        confident_target_size = 0
        random_target_size = 0
        threshold_mean = np.nan
        current_effective_mean_weight = np.nan
        replay_effective_mean_weight = np.nan
        replay_class_weight_dict = {0: np.nan, 1: np.nan}
        replay_weight_fallback = 'no_replay'

        for epoch in range(MAX_EPOCHS):
            if replay_used_size > 0:
                confident_target_size = int(np.floor(replay_used_size * CONFIDENTLY_CORRECT_REPLAY_FRACTION))
                random_target_size = int(replay_used_size - confident_target_size)

                confident_indices, threshold_values = sample_confidently_correct_replay_indices(
                    model=model,
                    X_replay_pool=X_replay_pool,
                    y_replay_pool=y_replay_pool,
                    replay_year_spans=replay_year_spans,
                    confident_target_size=confident_target_size,
                    rng=replay_rng,
                    threshold_grid=THRESHOLD_GRID,
                )
                random_indices = sample_random_replay_indices(
                    replay_pool_size=replay_pool_size,
                    random_target_size=random_target_size,
                    excluded_indices=confident_indices,
                    rng=replay_rng,
                )

                replay_indices = np.concatenate([confident_indices, random_indices])
                if len(replay_indices) > 0:
                    replay_indices = replay_indices.astype(np.int64, copy=False)
                    replay_indices = replay_indices[replay_rng.permutation(len(replay_indices))]

                confident_used_size = int(len(confident_indices))
                random_used_size = int(len(random_indices))
                if threshold_values:
                    threshold_mean = float(np.mean(threshold_values))
                else:
                    threshold_mean = np.nan

                if len(replay_indices) > 0:
                    X_replay_sampled = X_replay_pool[replay_indices]
                    y_replay_sampled = y_replay_pool[replay_indices]
                    replay_actual_positive_rate = float(np.mean(y_replay_sampled == REPLAY_POSITIVE_LABEL))
                    replay_class_weight_dict, replay_weight_fallback = compute_binary_class_weights(
                        y_replay_sampled,
                        classes=classes,
                        fallback_mode=REPLAY_WEIGHT_FALLBACK,
                    )

                    X_combined = np.concatenate([X_train_batch, X_replay_sampled], axis=0)
                    y_combined = np.concatenate([y_train_batch, y_replay_sampled], axis=0)
                    origin_is_replay_combined = np.concatenate(
                        [
                            np.zeros(len(y_train_batch), dtype=bool),
                            np.ones(len(y_replay_sampled), dtype=bool),
                        ]
                    )
                    replay_used_size = int(len(y_replay_sampled))
                else:
                    X_combined = X_train_batch
                    y_combined = y_train_batch
                    origin_is_replay_combined = np.zeros(len(y_train_batch), dtype=bool)
                    replay_used_size = 0
                    replay_actual_positive_rate = np.nan
                    replay_class_weight_dict = {0: np.nan, 1: np.nan}
                    replay_weight_fallback = 'no_replay'
            else:
                X_combined = X_train_batch
                y_combined = y_train_batch
                origin_is_replay_combined = np.zeros(len(y_train_batch), dtype=bool)
                replay_used_size = 0
                replay_actual_positive_rate = np.nan
                replay_class_weight_dict = {0: np.nan, 1: np.nan}
                replay_weight_fallback = 'no_replay'

            combined_n_samples = len(X_combined)
            shuffle_idx = replay_rng.permutation(combined_n_samples)
            X_train_shuffled = X_combined[shuffle_idx]
            y_train_shuffled = y_combined[shuffle_idx]
            origin_is_replay_shuffled = origin_is_replay_combined[shuffle_idx]
            n_chunks = max(1, int(np.ceil(combined_n_samples / CHUNK_SIZE)))

            for chunk_idx in range(n_chunks):
                start_idx = chunk_idx * CHUNK_SIZE
                end_idx = min(start_idx + CHUNK_SIZE, combined_n_samples)
                X_chunk = X_train_shuffled[start_idx:end_idx]
                y_chunk = y_train_shuffled[start_idx:end_idx]
                origin_is_replay_chunk = origin_is_replay_shuffled[start_idx:end_idx]

                sample_weights_chunk = np.empty(len(y_chunk), dtype=np.float64)
                current_chunk_mask = ~origin_is_replay_chunk
                replay_chunk_mask = origin_is_replay_chunk
                if np.any(current_chunk_mask):
                    sample_weights_chunk[current_chunk_mask] = np.array(
                        [current_class_weight_dict[int(label)] for label in y_chunk[current_chunk_mask]],
                        dtype=np.float64,
                    )
                if np.any(replay_chunk_mask):
                    sample_weights_chunk[replay_chunk_mask] = np.array(
                        [replay_class_weight_dict[int(label)] for label in y_chunk[replay_chunk_mask]],
                        dtype=np.float64,
                    )

                if len(sample_weights_chunk) != len(y_chunk):
                    raise ValueError(
                        f'Sample weight length mismatch: weights={len(sample_weights_chunk)} vs labels={len(y_chunk)}'
                    )
                if np.any(~np.isfinite(sample_weights_chunk)):
                    raise ValueError('Encountered non-finite sample weights in training chunk.')
                if np.any(sample_weights_chunk <= 0):
                    raise ValueError('Encountered non-positive sample weights in training chunk.')

                model.partial_fit(X_chunk, y_chunk, classes=classes, sample_weight=sample_weights_chunk)

            current_mask = ~origin_is_replay_shuffled
            replay_mask = origin_is_replay_shuffled
            if np.any(current_mask):
                current_effective_mean_weight = float(
                    np.mean(
                        np.array([current_class_weight_dict[int(label)] for label in y_train_shuffled[current_mask]], dtype=np.float64)
                    )
                )
            if np.any(replay_mask):
                replay_effective_mean_weight = float(
                    np.mean(
                        np.array([replay_class_weight_dict[int(label)] for label in y_train_shuffled[replay_mask]], dtype=np.float64)
                    )
                )

            y_val_pred = model.predict(X_val_batch)
            y_val_proba = model.predict_proba(X_val_batch)[:, 1]
            val_pr_auc = average_precision_score(y_val_batch, y_val_proba) if len(np.unique(y_val_batch)) > 1 else np.nan

            if val_pr_auc > best_val_pr_auc + MIN_DELTA:
                best_val_pr_auc = val_pr_auc
                patience_counter = 0
                best_model_state = {
                    'coefs': [w.copy() for w in model.coefs_],
                    'intercepts': [b.copy() for b in model.intercepts_],
                    'n_layers_': model.n_layers_,
                    'n_outputs_': getattr(model, 'n_outputs_', None),
                    'out_activation_': getattr(model, 'out_activation_', None),
                }
            else:
                patience_counter += 1
                if patience_counter >= PATIENCE:
                    if best_model_state is not None:
                        model.coefs_ = [w.copy() for w in best_model_state['coefs']]
                        model.intercepts_ = [b.copy() for b in best_model_state['intercepts']]
                        model.n_layers_ = best_model_state['n_layers_']
                        if best_model_state['n_outputs_'] is not None:
                            model.n_outputs_ = best_model_state['n_outputs_']
                        if best_model_state['out_activation_'] is not None:
                            model.out_activation_ = best_model_state['out_activation_']
                    break

        y_train_pred = model.predict(X_train_batch)
        y_val_pred = model.predict(X_val_batch)
        y_val_proba = model.predict_proba(X_val_batch)[:, 1]

        train_acc = accuracy_score(y_train_batch, y_train_pred)
        train_prec = precision_score(y_train_batch, y_train_pred, zero_division=0)
        train_rec = recall_score(y_train_batch, y_train_pred, zero_division=0)
        train_f1 = f1_score(y_train_batch, y_train_pred, zero_division=0)

        val_acc = accuracy_score(y_val_batch, y_val_pred)
        val_prec = precision_score(y_val_batch, y_val_pred, zero_division=0)
        val_rec = recall_score(y_val_batch, y_val_pred, zero_division=0)
        val_f1 = f1_score(y_val_batch, y_val_pred, zero_division=0)
        if len(np.unique(y_val_batch)) > 1:
            val_roc_auc = roc_auc_score(y_val_batch, y_val_proba)
            val_pr_auc = average_precision_score(y_val_batch, y_val_proba)
        else:
            val_roc_auc = np.nan
            val_pr_auc = np.nan

        training_history['year'].append(year_val)
        training_history['train_accuracy'].append(train_acc)
        training_history['train_precision'].append(train_prec)
        training_history['train_recall'].append(train_rec)
        training_history['train_f1'].append(train_f1)
        training_history['val_accuracy'].append(val_acc)
        training_history['val_precision'].append(val_prec)
        training_history['val_recall'].append(val_rec)
        training_history['val_f1'].append(val_f1)
        training_history['val_roc_auc'].append(val_roc_auc)
        training_history['val_pr_auc'].append(val_pr_auc)
        training_history['replay_pool_size'].append(int(replay_pool_size))
        training_history['replay_target_size'].append(int(replay_target_size))
        training_history['replay_used_size'].append(int(replay_used_size))
        training_history['confidently_correct_target_size'].append(int(confident_target_size))
        training_history['confidently_correct_used_size'].append(int(confident_used_size))
        training_history['random_target_size'].append(int(random_target_size))
        training_history['random_used_size'].append(int(random_used_size))
        training_history['threshold_mean'].append(float(threshold_mean) if not np.isnan(threshold_mean) else np.nan)
        training_history['weight_policy'].append(WEIGHT_POLICY_NAME)
        training_history['year_positive_rate'].append(float(year_positive_rate))
        training_history['replay_actual_positive_rate'].append(float(replay_actual_positive_rate) if replay_used_size > 0 else np.nan)
        training_history['replay_pool_positive_rate'].append(float(replay_pool_positive_rate) if replay_pool_size > 0 else np.nan)
        training_history['current_class_weight_0'].append(float(current_class_weight_dict[0]))
        training_history['current_class_weight_1'].append(float(current_class_weight_dict[1]))
        training_history['replay_class_weight_0'].append(float(replay_class_weight_dict[0]) if replay_used_size > 0 else np.nan)
        training_history['replay_class_weight_1'].append(float(replay_class_weight_dict[1]) if replay_used_size > 0 else np.nan)
        training_history['replay_weight_fallback'].append(replay_weight_fallback)

        year_model_path = models_dir / model_file_template.format(year=year_val)
        with open(year_model_path, 'wb') as f:
            pickle.dump(model, f)

        completed_years.add(year_val)
        completion_status['completed_years'][ratio_key] = sorted(list(completed_years))
        all_training_histories[ratio_key] = training_history.copy()
        save_all_training_histories(ALL_HISTORIES_FILE, all_training_histories)
        save_completion_status(COMPLETION_STATUS_FILE, completion_status)

        print(
            f'[{ratio_key}] Year {year_val}: Train F1={train_f1:.3f}, Val F1={val_f1:.3f}, Val PR-AUC={val_pr_auc:.3f}, '
            f'replay_used={replay_used_size:,}/{replay_pool_size:,}, replay_pos_rate={replay_actual_positive_rate:.2%}, '
            f'year_pos_rate={year_positive_rate:.2%}, replay_w0={replay_class_weight_dict[0]:.3f}, replay_w1={replay_class_weight_dict[1]:.3f}'
        )
        append_training_log(
            TRAINING_LOG_FILE,
            f'[{ratio_key}] completed year {year_val} with replay_used={replay_used_size}/{replay_pool_size}, '
            f'confidently_correct={confident_used_size}, random={random_used_size}, replay_pos_rate={replay_actual_positive_rate:.4f}, '
            f'year_pos_rate={year_positive_rate:.4f}, replay_pool_pos_rate={replay_pool_positive_rate:.4f}, '
            f'current_w0={current_class_weight_dict[0]:.4f}, current_w1={current_class_weight_dict[1]:.4f}, '
            f'replay_w0={replay_class_weight_dict[0]:.4f}, replay_w1={replay_class_weight_dict[1]:.4f}, '
            f'replay_weight_fallback={replay_weight_fallback}, '
            f'current_effective_mean_weight={current_effective_mean_weight:.4f}, replay_effective_mean_weight={replay_effective_mean_weight:.4f}.',
        )

    if hasattr(incremental_scaler, 'mean_'):
        final_scaler_path = models_dir / final_scaler_filename
        with open(final_scaler_path, 'wb') as f:
            pickle.dump(incremental_scaler, f)
        print(f'[{ratio_key}] Final incremental scaler saved: {final_scaler_path.name}')

    final_model_path = models_dir / final_model_filename
    with open(final_model_path, 'wb') as f:
        pickle.dump(model, f)
    print(f'[{ratio_key}] Final model saved: {final_model_path.name}')

    ratio_history_df = pd.DataFrame(training_history).sort_values('year').reset_index(drop=True)
    ratio_history_path = models_dir / history_filename
    ratio_history_df.to_csv(ratio_history_path, index=False)
    print(f'[{ratio_key}] History saved: {ratio_history_path}')

    if set(all_target_year_values).issubset(completed_years):
        if ratio_key not in completion_status.get('completed_ratios', []):
            completion_status.setdefault('completed_ratios', []).append(ratio_key)
        save_completion_status(COMPLETION_STATUS_FILE, completion_status)
        append_training_log(TRAINING_LOG_FILE, f'[{ratio_key}] marked as fully completed.')

    all_training_histories[ratio_key] = training_history.copy()
    save_all_training_histories(ALL_HISTORIES_FILE, all_training_histories)

combined_history_frames = []
for ratio_key, history_dict in all_training_histories.items():
    if not isinstance(history_dict, dict):
        continue
    if len(history_dict.get('year', [])) == 0:
        continue
    df_ratio = pd.DataFrame(history_dict).sort_values('year').reset_index(drop=True)
    replay_ratio = float(ratio_key.split('_')[1])
    df_ratio['ratio_key'] = ratio_key
    df_ratio['replay_ratio'] = replay_ratio
    combined_history_frames.append(df_ratio)

if combined_history_frames:
    combined_history_df = pd.concat(combined_history_frames, ignore_index=True)
    combined_history_df = combined_history_df.sort_values(['replay_ratio', 'year']).reset_index(drop=True)
    combined_history_path = Path('.') / COMBINED_HISTORY_FILENAME
    combined_history_df.to_csv(combined_history_path, index=False)
    print(f'Combined history saved: {combined_history_path}')
    print(f'Combined rows: {len(combined_history_df):,}')
else:
    print('No history data available to write combined history.')

save_completion_status(COMPLETION_STATUS_FILE, completion_status)
save_all_training_histories(ALL_HISTORIES_FILE, all_training_histories)
append_training_log(TRAINING_LOG_FILE, 'Training run completed.')

Checkpoint directory: C:\Users\bartu\Desktop\Fonda-scikit - Git\training_checkpoints_mlp_experience_replay_confidently_correct_memory
Replay ratios: [0.2, 0.3, 0.4, 0.5]
Weight policy: post_sampling_replay_class_weights_v1, replay_weight_fallback=smoothed_single_class
[RR_0.2] already completed. Skipping ratio.
[RR_0.3] already completed. Skipping ratio.
[RR_0.4] already completed. Skipping ratio.
[RR_0.5] already completed. Skipping ratio.
Combined history saved: mlp_classifier_history_prevyears_monthly_features_incremental_scaler_experience_replay_confidently_correct_memory_all_ratios.csv
Combined rows: 24


## 5. Save Training History

In [6]:
import json

CHECKPOINT_DIR = Path('.') / 'training_checkpoints_mlp_experience_replay_confidently_correct_memory'
ALL_HISTORIES_FILE = CHECKPOINT_DIR / 'mlp_replay_all_training_histories_confidently_correct_memory.pkl'
COMPLETION_STATUS_FILE = CHECKPOINT_DIR / 'mlp_replay_completion_status_confidently_correct_memory.json'
COMBINED_HISTORY_FILENAME = 'mlp_classifier_history_prevyears_monthly_features_incremental_scaler_experience_replay_confidently_correct_memory_all_ratios.csv'

if ALL_HISTORIES_FILE.exists():
    with open(ALL_HISTORIES_FILE, 'rb') as f:
        all_training_histories = pickle.load(f)
else:
    all_training_histories = {}

if COMPLETION_STATUS_FILE.exists():
    with open(COMPLETION_STATUS_FILE, 'r', encoding='utf-8') as f:
        completion_status = json.load(f)
else:
    completion_status = {'completed_ratios': [], 'completed_years': {}}

summary_rows = []
for ratio_key in sorted(all_training_histories.keys()):
    history_dict = all_training_histories[ratio_key]
    n_rows = len(history_dict.get('year', []))
    last_year = history_dict['year'][-1] if n_rows > 0 else np.nan
    last_val_f1 = history_dict['val_f1'][-1] if n_rows > 0 else np.nan
    last_val_pr_auc = history_dict['val_pr_auc'][-1] if n_rows > 0 else np.nan
    is_completed = ratio_key in completion_status.get('completed_ratios', [])
    summary_rows.append(
        {
            'ratio_key': ratio_key,
            'rows': n_rows,
            'last_year': last_year,
            'last_val_f1': last_val_f1,
            'last_val_pr_auc': last_val_pr_auc,
            'completed': is_completed,
        }
    )

summary_df = pd.DataFrame(summary_rows).sort_values('ratio_key').reset_index(drop=True)
print('Per-ratio training summary:')
display(summary_df)

combined_history_path = Path('.') / COMBINED_HISTORY_FILENAME
if combined_history_path.exists():
    combined_df = pd.read_csv(combined_history_path)
    print(f'Combined history file: {combined_history_path}')
    print(f'Rows: {len(combined_df):,}')
    display(combined_df.tail())
else:
    print(f'Combined history file not found: {combined_history_path}')

Per-ratio training summary:


,ratio_key,rows,last_year,last_val_f1,last_val_pr_auc,completed
0,RR_0.2,6,2022,0.237094,0.353745,True
1,RR_0.3,6,2022,0.252110,0.346925,True
2,RR_0.4,6,2022,0.224611,0.350414,True
3,RR_0.5,6,2022,0.229688,0.345543,True


Combined history file: mlp_classifier_history_prevyears_monthly_features_incremental_scaler_experience_replay_confidently_correct_memory_all_ratios.csv
Rows: 24


,year,train_accuracy,train_precision,train_recall,train_f1,val_accuracy,val_precision,val_recall,val_f1,val_roc_auc,...,year_positive_rate,replay_actual_positive_rate,replay_pool_positive_rate,current_class_weight_0,current_class_weight_1,replay_class_weight_0,replay_class_weight_1,replay_weight_fallback,ratio_key,replay_ratio
19,2018,0.902722,0.145418,0.766056,0.244436,0.895602,0.141368,0.732355,0.236990,0.901448,...,0.020541,0.017105,0.018464,0.510486,24.341689,0.508701,29.230704,balanced,RR_0.5,0.5
20,2019,0.892898,0.131557,0.765588,0.224531,0.898742,0.122702,0.738158,0.210425,0.902609,...,0.020253,0.017749,0.019504,0.510336,24.688109,0.509035,28.169914,balanced,RR_0.5,0.5
21,2020,0.907211,0.137280,0.753346,0.232239,0.911400,0.126622,0.718416,0.215298,0.904037,...,0.018629,0.018120,0.019753,0.509491,26.840525,0.509227,27.593149,balanced,RR_0.5,0.5
22,2021,0.899898,0.108724,0.774735,0.190687,0.893195,0.103967,0.768974,0.183169,0.906662,...,0.015222,0.017696,0.019472,0.507729,32.847622,0.509007,28.255250,balanced,RR_0.5,0.5
23,2022,0.859045,0.153994,0.800899,0.258319,0.860950,0.136670,0.719130,0.229688,0.883713,...,0.030649,0.016809,0.018622,0.515809,16.313904,0.508548,29.746372,balanced,RR_0.5,0.5
